In [1]:
import numpy as np
import keras
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv('datasets/insurance_data.csv')
df.head()

,age,affordibility,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1


In [61]:
df.shape

(28, 3)

In [62]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df[['age', 'affordibility']], df['bought_insurance'], test_size=0.2, random_state=42)

In [63]:
len(X_train)

22

In [64]:
X_train_scaled = X_train.copy()
X_train_scaled['age'] = X_train_scaled['age']/100

x_test_scaled = X_test.copy()
x_test_scaled['age'] = x_test_scaled['age']/100

In [65]:
X_train_scaled.head()

,age,affordibility
17,0.58,1
22,0.40,1
11,0.28,1
13,0.29,0
15,0.55,1


In [8]:
model = keras.Sequential([
    keras.layers.Dense(1, input_shape=(2,), activation='sigmoid', kernel_initializer='ones', bias_initializer='zeros')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_scaled, y_train, epochs=5000, verbose=0)

c:\Users\adabh\OneDrive\Desktop\vknox\ml_learning\ml\.venv\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.evaluate(x_test_scaled, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 1.0000 - loss: 0.2647


[0.26470306515693665, 1.0]

In [66]:
x_test_scaled.head()

,age,affordibility
9,0.61,1
25,0.54,1
8,0.62,1
21,0.26,0
0,0.22,1


In [67]:
model.predict(x_test_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step


array([[0.81626713],
       [0.7576754 ],
       [0.82367426],
       [0.18819512],
       [0.38559145],
       [0.19598222]], dtype=float32)

In [68]:
y_test.head()

9     1
25    1
8     1
21    0
0     0
Name: bought_insurance, dtype: int64

In [69]:
coef, intercept = model.get_weights()
coef, intercept

(array([[5.018309 ],
        [1.1966311]], dtype=float32),
 array([-2.766541], dtype=float32))

In [70]:
def sigmoid(x):
    import math
    return 1 / (1 + np.exp(-x))
sigmoid(18)

np.float64(0.9999999847700205)

In [71]:
def prediction_function(age, affordibility):
    z = coef[0] * age + coef[1] * affordibility + intercept
    return sigmoid(z)

In [72]:
prediction_function(0.61, 1)

array([0.8162671], dtype=float32)

In [73]:
def log_loss(y_actual, y_predicted):
    epsilon = 1e-15
    y_predicted = [max(i, epsilon) for i in y_predicted]
    y_predicted_new = [min(i, 1 - epsilon) for i in y_predicted]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_actual * np.log(y_predicted_new) + (1 - y_actual) * np.log(1 - y_predicted_new))

In [74]:
def sigmoid_numpy(x):
    return 1 / (1 + np.exp(-x))

sigmoid_numpy(np.array([18, 20, 25]))

array([0.99999998, 1.        , 1.        ])

In [75]:
# gradient descent function
def gradient_descent(age, affordibility, y_actual, epochs, loss_threshold):
    # Initialize weights and bias
    w1 = w2 = 1
    bias = 0
    learning_rate = 0.5
    n = len(age)
    
    for i in range(epochs):
        # Calculate predictions
        z = w1 * age + w2 * affordibility + bias
        y_predicted = sigmoid_numpy(z)
        
        loss = log_loss(y_actual, y_predicted)
        # Calculate gradients
        dw1 = (1/n) * np.dot(np.transpose(y_predicted - y_actual), age)
        dw2 = (1/n) * np.dot(np.transpose(y_predicted - y_actual), affordibility)
        dbias = (1/n) * np.sum(y_predicted - y_actual)
        
        # Update weights and bias
        w1 -= learning_rate * dw1
        w2 -= learning_rate * dw2
        bias -= learning_rate * dbias
        
        print(f'Epoch {i+1}/{epochs}, Loss: {loss:.4f}, w1: {w1:.4f}, w2: {w2:.4f}, bias: {bias:.4f}')
        
        if loss <= loss_threshold:
            print(f'Loss threshold reached at epoch {i+1}. Stopping training.')
            break
    return w1, w2, bias

In [76]:
gradient_descent(X_train_scaled['age'], X_train_scaled['affordibility'], y_train, 1000, 0.446)

Epoch 1/1000, Loss: 0.7428, w1: 0.9737, w2: 0.9314, bias: -0.1175
Epoch 2/1000, Loss: 0.7072, w1: 0.9537, w2: 0.8740, bias: -0.2188
Epoch 3/1000, Loss: 0.6815, w1: 0.9394, w2: 0.8272, bias: -0.3054
Epoch 4/1000, Loss: 0.6633, w1: 0.9302, w2: 0.7898, bias: -0.3788
Epoch 5/1000, Loss: 0.6507, w1: 0.9254, w2: 0.7606, bias: -0.4411
Epoch 6/1000, Loss: 0.6421, w1: 0.9243, w2: 0.7383, bias: -0.4938
Epoch 7/1000, Loss: 0.6360, w1: 0.9263, w2: 0.7218, bias: -0.5387
Epoch 8/1000, Loss: 0.6318, w1: 0.9309, w2: 0.7101, bias: -0.5772
Epoch 9/1000, Loss: 0.6288, w1: 0.9374, w2: 0.7022, bias: -0.6103
Epoch 10/1000, Loss: 0.6265, w1: 0.9457, w2: 0.6973, bias: -0.6392
Epoch 11/1000, Loss: 0.6248, w1: 0.9552, w2: 0.6949, bias: -0.6646
Epoch 12/1000, Loss: 0.6234, w1: 0.9659, w2: 0.6945, bias: -0.6872
Epoch 13/1000, Loss: 0.6222, w1: 0.9774, w2: 0.6956, bias: -0.7076
Epoch 14/1000, Loss: 0.6211, w1: 0.9896, w2: 0.6979, bias: -0.7262
Epoch 15/1000, Loss: 0.6201, w1: 1.0023, w2: 0.7012, bias: -0.7432
Epoc

(np.float64(8.27941529517949),
 np.float64(1.4885842699102663),
 np.float64(-4.322622277165126))

In [77]:
coef, intercept

(array([[5.018309 ],
        [1.1966311]], dtype=float32),
 array([-2.766541], dtype=float32))

### Implementing Neural Network From Scratch

In [78]:
class myNN:
    def __init__(self):
        self.w1 = 1
        self.w2 = 1
        self.bias = 0
        
    def fit(self, X, y, epochs, loss_threshold):
        self.w1, self.w2, self.bias = self.gradient_descent(X['age'],X['affordibility'], y, epochs, loss_threshold)
        
    def predict(self, X_test):
        z = self.w1 * X_test['age'] + self.w2 * X_test['affordibility'] + self.bias
        return sigmoid_numpy(z)

    def gradient_descent(self, age, affordibility, y_actual, epochs, loss_threshold):
    # Initialize weights and bias
        w1 = w2 = 1
        bias = 0
        learning_rate = 0.01
        n = len(age)
        
        for i in range(epochs):
            # Calculate predictions
            z = w1 * age + w2 * affordibility + bias
            y_predicted = sigmoid_numpy(z)
            
            loss = log_loss(y_actual, y_predicted)
            # Calculate gradients
            dw1 = (1/n) * np.dot(np.transpose(y_predicted - y_actual), age)
            dw2 = (1/n) * np.dot(np.transpose(y_predicted - y_actual), affordibility)
            dbias = (1/n) * np.sum(y_predicted - y_actual)
            
            # Update weights and bias
            w1 -= learning_rate * dw1
            w2 -= learning_rate * dw2
            bias -= learning_rate * dbias
            
            if i % 50 == 0:
                print(f'Epoch {i+1}/{epochs}, Loss: {loss:.4f}, w1: {w1:.4f}, w2: {w2:.4f}, bias: {bias:.4f}')
            
            if loss <= loss_threshold:
                print(f'Loss threshold reached at epoch {i+1}. Stopping training.')
                break
        return w1, w2, bias

In [79]:
myModel = myNN()
myModel.fit(X_train_scaled, y_train, epochs=1000, loss_threshold=0.503)

Epoch 1/1000, Loss: 0.7428, w1: 0.9995, w2: 0.9986, bias: -0.0023
Epoch 51/1000, Loss: 0.7094, w1: 0.9762, w2: 0.9354, bias: -0.1121
Epoch 101/1000, Loss: 0.6848, w1: 0.9584, w2: 0.8822, bias: -0.2074
Epoch 151/1000, Loss: 0.6670, w1: 0.9458, w2: 0.8384, bias: -0.2898
Epoch 201/1000, Loss: 0.6542, w1: 0.9376, w2: 0.8028, bias: -0.3607
Epoch 251/1000, Loss: 0.6450, w1: 0.9333, w2: 0.7745, bias: -0.4216
Epoch 301/1000, Loss: 0.6385, w1: 0.9324, w2: 0.7524, bias: -0.4741
Epoch 351/1000, Loss: 0.6338, w1: 0.9342, w2: 0.7354, bias: -0.5194
Epoch 401/1000, Loss: 0.6303, w1: 0.9383, w2: 0.7229, bias: -0.5588
Epoch 451/1000, Loss: 0.6277, w1: 0.9443, w2: 0.7139, bias: -0.5932
Epoch 501/1000, Loss: 0.6256, w1: 0.9520, w2: 0.7079, bias: -0.6235
Epoch 551/1000, Loss: 0.6240, w1: 0.9610, w2: 0.7043, bias: -0.6504
Epoch 601/1000, Loss: 0.6226, w1: 0.9710, w2: 0.7026, bias: -0.6744
Epoch 651/1000, Loss: 0.6214, w1: 0.9819, w2: 0.7026, bias: -0.6962
Epoch 701/1000, Loss: 0.6204, w1: 0.9936, w2: 0.703

In [80]:
myModel.predict(X_test)

9     1.0
25    1.0
8     1.0
21    1.0
0     1.0
12    1.0
dtype: float64

Something's wrong with this. Let's see i think scaling is not right and learning rate is high due to which it is converging faster, vanishing gradient.

In [81]:
import numpy as np

def sigmoid_numpy(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -(1/len(y_true)) * np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

class myNN:
    def __init__(self):
        # Glorot initialization
        limit = np.sqrt(6 / (2 + 1))
        self.w1 = np.random.uniform(-limit, limit)
        self.w2 = np.random.uniform(-limit, limit)
        self.bias = 0.0

    def fit(self, X, y, epochs):
        # Preprocess data (standardize)
        X = X.copy()
        X['age'] = (X['age'] - X['age'].mean()) / X['age'].std()
        X['affordibility'] = (X['affordibility'] - X['affordibility'].mean()) / X['affordibility'].std()
        y = y.astype(np.float32)
        # Train
        self.w1, self.w2, self.bias = self.gradient_descent(X['age'], X['affordibility'], y, epochs)

    def predict(self, X_test):
        # Preprocess test data
        X_test = X_test.copy()
        X_test['age'] = (X_test['age'] - X_test['age'].mean()) / X_test['age'].std()
        X_test['affordibility'] = (X_test['affordibility'] - X_test['affordibility'].mean()) / X_test['affordibility'].std()
        z = self.w1 * X_test['age'] + self.w2 * X_test['affordibility'] + self.bias
        return sigmoid_numpy(z)

    def gradient_descent(self, age, affordibility, y_actual, epochs):
        age = age.astype(np.float32)
        affordibility = affordibility.astype(np.float32)
        y_actual = y_actual.astype(np.float32)
        w1 = np.float32(self.w1)
        w2 = np.float32(self.w2)
        bias = np.float32(self.bias)
        learning_rate = 0.01  # Reduced from 0.5
        n = len(age)

        for i in range(epochs):
            z = w1 * age + w2 * affordibility + bias
            y_predicted = sigmoid_numpy(z)
            loss = log_loss(y_actual, y_predicted)

            # Gradients
            error = y_predicted - y_actual
            dw1 = (1/n) * np.dot(error, age)
            dw2 = (1/n) * np.dot(error, affordibility)
            dbias = (1/n) * np.sum(error)

            # Update weights
            w1 -= learning_rate * dw1
            w2 -= learning_rate * dw2
            bias -= learning_rate * dbias

            if i % 50 == 0:
                print(f'Epoch {i+1}/{epochs}, Loss: {loss:.4f}, w1: {w1:.4f}, w2: {w2:.4f}, bias: {bias:.4f}')

            # if loss <= loss_threshold:
            #     print(f'Loss threshold reached at epoch {i+1}. Stopping training.')
            #     break

        return w1, w2, bias

In [82]:
myModel = myNN()
myModel.fit(X_train_scaled, y_train, epochs=1000)

Epoch 1/1000, Loss: 0.5365, w1: 0.4991, w2: 0.4045, bias: -0.0000
Epoch 51/1000, Loss: 0.5205, w1: 0.5855, w2: 0.4265, bias: -0.0012
Epoch 101/1000, Loss: 0.5075, w1: 0.6635, w2: 0.4467, bias: -0.0025
Epoch 151/1000, Loss: 0.4968, w1: 0.7340, w2: 0.4653, bias: -0.0040
Epoch 201/1000, Loss: 0.4880, w1: 0.7981, w2: 0.4826, bias: -0.0057
Epoch 251/1000, Loss: 0.4806, w1: 0.8564, w2: 0.4988, bias: -0.0076
Epoch 301/1000, Loss: 0.4745, w1: 0.9097, w2: 0.5140, bias: -0.0097
Epoch 351/1000, Loss: 0.4692, w1: 0.9586, w2: 0.5284, bias: -0.0119
Epoch 401/1000, Loss: 0.4648, w1: 1.0034, w2: 0.5421, bias: -0.0142
Epoch 451/1000, Loss: 0.4611, w1: 1.0447, w2: 0.5551, bias: -0.0166
Epoch 501/1000, Loss: 0.4578, w1: 1.0828, w2: 0.5674, bias: -0.0192
Epoch 551/1000, Loss: 0.4550, w1: 1.1181, w2: 0.5792, bias: -0.0218
Epoch 601/1000, Loss: 0.4526, w1: 1.1508, w2: 0.5905, bias: -0.0245
Epoch 651/1000, Loss: 0.4505, w1: 1.1811, w2: 0.6014, bias: -0.0272
Epoch 701/1000, Loss: 0.4487, w1: 1.2093, w2: 0.611

In [83]:
myModel.predict(X_test)

9     0.849792
25    0.774799
8     0.858638
21    0.114737
0     0.261529
12    0.122154
dtype: float64

It's fine now, 9, 25 and 8 have greater probabilities that 0.5, so sigmoid will output true label